# Ingestion PDF

## Loader sait:
### óu trouver le fichier 
### comment l'ouvrir
### comment lire sa structure interne
### et comment en extraire le texte

In [1]:
from langchain_community.document_loaders import PyPDFLoader

/tmp/ipykernel_148477/4175148793.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/home/eg/Schreibtisch/RAG-Projekt/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pdf_path = "../data/pdf_files/GWB.pdf"

In [3]:
loader = PyPDFLoader(pdf_path)
pdf_docs = loader.load()

In [4]:
print("nombre de pages dans le PDF :", len(pdf_docs))
print(pdf_docs[0].page_content[:500], "...")
print(pdf_docs[0].metadata)

nombre de pages dans le PDF : 126
Ein Service des Bundesministeriums der Justiz sowie des Bundesamts für
Justiz ‒ www.gesetze-im-internet.de
Gesetz gegen Wettbewerbsbeschränkungen (GWB)
GWB
Ausfertigungsdatum: 26.08.1998
Vollzitat:
"Gesetz gegen Wettbewerbsbeschränkungen in der Fassung der Bekanntmachung vom 26. Juni 2013 (BGBl. I S.
1750, 3245), das zuletzt durch Artikel 6 des Gesetzes vom 5. Dezember 2024 (BGBl. 2024 I Nr. 400) geändert
worden ist"
Stand: Neugefasst durch Bek. v. 26.6.2013 I 1750, 3245;
  zuletzt geändert durc ...
{'producer': 'Apache FOP Version 2.6', 'creator': 'Apache FOP Version 2.6', 'creationdate': '2024-12-31T22:57:10+01:00', 'source': '../data/pdf_files/GWB.pdf', 'total_pages': 126, 'page': 0, 'page_label': '1'}


# ingestion avancée avec PyMUPDFLOADER

In [5]:
from langchain_community.document_loaders import PyMuPDFLoader
pdf_path = "../data/pdf_files/GWB.pdf"
loader2 = PyMuPDFLoader(pdf_path)
pdfmu_docs = loader2.load()

print("nombre de pages dans le PDF :", len(pdfmu_docs))
print(pdfmu_docs[0].page_content[:500], "...")
print(pdfmu_docs[0].metadata)

nombre de pages dans le PDF : 126
Ein Service des Bundesministeriums der Justiz sowie des Bundesamts für
Justiz ‒ www.gesetze-im-internet.de
Gesetz gegen Wettbewerbsbeschränkungen (GWB)
GWB
Ausfertigungsdatum: 26.08.1998
Vollzitat:
"Gesetz gegen Wettbewerbsbeschränkungen in der Fassung der Bekanntmachung vom 26. Juni 2013 (BGBl. I S.
1750, 3245), das zuletzt durch Artikel 6 des Gesetzes vom 5. Dezember 2024 (BGBl. 2024 I Nr. 400) geändert
worden ist"
Stand:
Neugefasst durch Bek. v. 26.6.2013 I 1750, 3245;
 
zuletzt geändert durc ...
{'producer': 'Apache FOP Version 2.6', 'creator': 'Apache FOP Version 2.6', 'creationdate': '2024-12-31T22:57:10+01:00', 'source': '../data/pdf_files/GWB.pdf', 'file_path': '../data/pdf_files/GWB.pdf', 'total_pages': 126, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': "D:20241231225710+01'00'", 'page': 0}


# Fonction de nettoyage 

In [6]:
import re
def clean_pdf_text(text : str) -> str:
    # Supprimer les caractères spéciaux et les espaces multiples
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub("utf-8", "", text)
    text = re.sub(r"page\d+\s*:\s*\d+", "", text, flags=re.IGNORECASE)
    return text.strip()

# Applequer le netoyage

In [7]:
cleaned_docs = []

for doc in pdfmu_docs:
    cleaned = clean_pdf_text(doc.page_content)
    doc.page_content = cleaned
    cleaned_docs.append(doc)

In [8]:
print("Avant")
print(pdfmu_docs[0].page_content[:500], "...")
print("Après")
print(cleaned_docs[0].page_content[:500], "...")

Avant
Ein Service des Bundesministeriums der Justiz sowie des Bundesamts für Justiz  wwwgesetzeiminternetde Gesetz gegen Wettbewerbsbeschränkungen GWB GWB Ausfertigungsdatum 26081998 Vollzitat Gesetz gegen Wettbewerbsbeschränkungen in der Fassung der Bekanntmachung vom 26 Juni 2013 BGBl I S 1750 3245 das zuletzt durch Artikel 6 des Gesetzes vom 5 Dezember 2024 BGBl 2024 I Nr 400 geändert worden ist Stand Neugefasst durch Bek v 2662013 I 1750 3245 zuletzt geändert durch Art 6 G v 5122024 I Nr 400 Änder ...
Après
Ein Service des Bundesministeriums der Justiz sowie des Bundesamts für Justiz  wwwgesetzeiminternetde Gesetz gegen Wettbewerbsbeschränkungen GWB GWB Ausfertigungsdatum 26081998 Vollzitat Gesetz gegen Wettbewerbsbeschränkungen in der Fassung der Bekanntmachung vom 26 Juni 2013 BGBl I S 1750 3245 das zuletzt durch Artikel 6 des Gesetzes vom 5 Dezember 2024 BGBl 2024 I Nr 400 geändert worden ist Stand Neugefasst durch Bek v 2662013 I 1750 3245 zuletzt geändert durch Art 6 G v 51220

# Chunking

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=["\n\n", "\n", " ", ""]
)

In [10]:
chunks = splitter.split_documents(cleaned_docs)
print("Nombre de chunks :", len(chunks))
print(chunks[1].page_content)

Nombre de chunks : 492
EGRL 812009 CELEX Nr 32009L0081 vgl Art 1 d G v 2532020 I 674 EURL 232014 CELEX Nr 32014L0023 vgl Art 1 d G v 2532020 I 674 und G v 25102023 I Nr 294 EURL 242014 CELEX Nr 32014L0024 vgl Art 1 d G v 2532020 I 674 EURL 252014 CELEX Nr 32014L0025 vgl Art 1 d G v 2532020 I 674 EWGRL 66589 CELEX Nr 31989L0665 vgl Art 1 d G v 2532020 I 674 EWGRL 1392 CELEX Nr 392L0013 vgl Art 1 d G v 2532020 I 674 EURL 20191 CELEX Nr 32019L0001 vgl Art 1 d G v 1812021 I 2 EURL 1042014 CELEX Nr 32014L0104 vgl Art 1 d G v 1812021 I 2 EGRL 722009 CELEX Nr 32009L0072 vgl G v 22122023 I Nr 405 EGRL 732009 CELEX Nr 32009L0073 vgl G v 22122023 I Nr 405 EURL 2019944 CELEX Nr 32019L0944 vgl G v 22122023 I Nr 405  Das G wurde als Artikel 1 G 70341 v 2681998 I 2521 WettbewGÄndG 6 vom Bundestag mit Zustimmung des Bundesrates beschlossen Es ist gem Art 4 dieses G am 111999 in Kraft getreten Inhaltsübersicht Teil 1  Seite 1 von 126


In [11]:
for i, c in enumerate(chunks):
    print(f"===Chunk {i}=== : {len(c.page_content)} caractères")
    print(c.page_content[:500], "...")

===Chunk 0=== : 1198 caractères
Ein Service des Bundesministeriums der Justiz sowie des Bundesamts für Justiz  wwwgesetzeiminternetde Gesetz gegen Wettbewerbsbeschränkungen GWB GWB Ausfertigungsdatum 26081998 Vollzitat Gesetz gegen Wettbewerbsbeschränkungen in der Fassung der Bekanntmachung vom 26 Juni 2013 BGBl I S 1750 3245 das zuletzt durch Artikel 6 des Gesetzes vom 5 Dezember 2024 BGBl 2024 I Nr 400 geändert worden ist Stand Neugefasst durch Bek v 2662013 I 1750 3245 zuletzt geändert durch Art 6 G v 5122024 I Nr 400 Änder ...
===Chunk 1=== : 908 caractères
EGRL 812009 CELEX Nr 32009L0081 vgl Art 1 d G v 2532020 I 674 EURL 232014 CELEX Nr 32014L0023 vgl Art 1 d G v 2532020 I 674 und G v 25102023 I Nr 294 EURL 242014 CELEX Nr 32014L0024 vgl Art 1 d G v 2532020 I 674 EURL 252014 CELEX Nr 32014L0025 vgl Art 1 d G v 2532020 I 674 EWGRL 66589 CELEX Nr 31989L0665 vgl Art 1 d G v 2532020 I 674 EWGRL 1392 CELEX Nr 392L0013 vgl Art 1 d G v 2532020 I 674 EURL 20191 CELEX Nr 32019L0001 vgl Ar